# 01 — Ingestion & EDA

This notebook ingests Uruguay OPP budget data from three sources, converts to Parquet, uploads to GCS, and performs exploratory data analysis.

**Sources:**
1. CKAN open data catalog — package search API (`catalogodatos.gub.uy`)
2. CKAN Datastore dump API — detailed budget credits 2011-2021 + 5-year budgets
3. PDF documents from `opp.gub.uy/es/presupuesto-nacional`

**Note:** The transparency portal (`transparenciapresupuestaria.opp.gub.uy`) blocks
direct file downloads (403), but the same data is available via the CKAN Datastore dump API.

**Outputs:** Parquet files uploaded to GCS bucket.

## Setup

In [ ]:
!pip install -q polars==1.24.0 httpx==0.28.1 beautifulsoup4==4.13.3 \
    google-cloud-storage==2.19.0 google-cloud-bigquery==3.27.0 \
    pdfplumber==0.11.6 pyarrow==18.1.0 lxml==5.3.1

In [ ]:
from google.colab import auth
auth.authenticate_user()

print("Authenticated successfully")

In [ ]:
import io
import time
from pathlib import Path
from urllib.parse import urljoin

import httpx
import polars as pl
from bs4 import BeautifulSoup
from google.cloud import storage

# ── Configuration ──
PROJECT_ID  = "fabled-imagery-488015-p6"
BUCKET_NAME = "opp-data-lake-fabled-imagery-488015-p6"

gcs_client = storage.Client(project=PROJECT_ID)
bucket     = gcs_client.bucket(BUCKET_NAME)


def read_csv_robust(raw_bytes: bytes, **kwargs) -> pl.DataFrame:
    """Read CSV with fallback for Latin American decimal comma format."""
    try:
        return pl.read_csv(
            io.BytesIO(raw_bytes), infer_schema_length=10000, **kwargs
        )
    except Exception:
        pass
    # Retry with decimal comma (Uruguayan locale: 208384,26)
    try:
        return pl.read_csv(
            io.BytesIO(raw_bytes), infer_schema_length=10000,
            decimal_comma=True, **kwargs
        )
    except Exception:
        pass
    # Last resort: read everything as strings
    return pl.read_csv(
        io.BytesIO(raw_bytes), infer_schema_length=0, **kwargs
    )


def upload_df_to_gcs(df: pl.DataFrame, blob_name: str) -> None:
    """Write a DataFrame as Parquet and upload to GCS."""
    buf = io.BytesIO()
    df.write_parquet(buf)
    buf.seek(0)
    blob = bucket.blob(blob_name)
    blob.upload_from_file(buf, content_type="application/octet-stream")


print(f"Project:  {PROJECT_ID}")
print(f"Bucket:   {BUCKET_NAME}")
print(f"GCS OK:   {bucket.exists()}")

---
## Stream A — CKAN Package Search

Query the CKAN API for all OPP-organization datasets, download CSV resources, convert to Parquet, upload to GCS.

In [ ]:
CKAN_API = "https://catalogodatos.gub.uy/api/3/action/package_search"

with httpx.Client() as client:
    resp = client.get(CKAN_API, params={"fq": "organization:opp", "rows": 50}, timeout=30.0)
    resp.raise_for_status()
    packages = resp.json()["result"]["results"]

print(f"Found {len(packages)} OPP packages")
for pkg in packages:
    formats = [r.get("format", "?") for r in pkg.get("resources", [])]
    print(f"  {pkg['name']}: {pkg['title'][:60]}  [{', '.join(formats)}]")

In [ ]:
# Extract CSV resource URLs — prefer datastore dump when available
DATASTORE_DUMP = "https://catalogodatos.gub.uy/datastore/dump"

csv_resources = []
for pkg in packages:
    for res in pkg.get("resources", []):
        fmt = (res.get("format") or "").upper()
        if fmt != "CSV" or not res.get("url"):
            continue

        # Use datastore dump if active (bypasses 403 on portal)
        if res.get("datastore_active"):
            url = f"{DATASTORE_DUMP}/{res['id']}?bom=True&format=csv"
        else:
            url = res["url"]

        csv_resources.append({
            "package_name": pkg["name"],
            "resource_id": res["id"],
            "url": url,
            "name": res.get("name", "unknown"),
            "via_datastore": res.get("datastore_active", False),
        })

ds_count = sum(1 for r in csv_resources if r["via_datastore"])
print(f"Found {len(csv_resources)} CSV resources ({ds_count} via datastore dump, {len(csv_resources) - ds_count} direct)")

In [ ]:
# Download CSVs → Parquet → GCS
ckan_dataframes = {}

with httpx.Client() as client:
    for res in csv_resources:
        blob_name = f"raw/ckan/{res['package_name']}_{res['resource_id']}.parquet"
        src = "DS" if res["via_datastore"] else "URL"

        try:
            r = client.get(res["url"], follow_redirects=True, timeout=60.0)
            r.raise_for_status()
            df = read_csv_robust(r.content)
        except Exception as exc:
            print(f"  SKIP [{src}] {res['name']}: {exc}")
            continue

        if df.is_empty():
            print(f"  SKIP [{src}] {res['name']}: empty")
            continue

        upload_df_to_gcs(df, blob_name)
        ckan_dataframes[res["name"]] = df
        print(f"  OK [{src}] {res['name']} → {df.shape[0]} rows, {df.shape[1]} cols")

print(f"\nUploaded {len(ckan_dataframes)} CKAN datasets to GCS")

---
## Stream B — CKAN Datastore: Detailed Budget Credits (2011–2021)

The transparency portal blocks direct downloads (403), but the **CKAN Datastore dump API**
serves the same data. These are the core budget datasets with yearly detail.

In [ ]:
DATASTORE_DUMP = "https://catalogodatos.gub.uy/datastore/dump"

# Key budget datasets available via datastore dump
BUDGET_RESOURCES = [
    {"name": "presupuesto_2020_2024", "id": "199524c6-faa1-4059-a608-bdea769b7d40",
     "description": "National Budget 2020-2024"},
    {"name": "presupuesto_2015_2019", "id": "e877b3d4-b61d-4989-9fa4-a3a4ae605ce7",
     "description": "National Budget 2015-2019"},
    {"name": "credito_resumen", "id": "97043c95-2c76-400a-81a2-22375e87e12a",
     "description": "Budget credit summary"},
    {"name": "credito_2021", "id": "a4ab3c46-8086-49e0-b971-79ab4d8127c5",
     "description": "Detailed budget credits 2021"},
    {"name": "credito_2020", "id": "1ddc58f5-6fdc-445c-9a52-97df3431268e",
     "description": "Detailed budget credits 2020"},
    {"name": "credito_2019", "id": "334e3597-3332-40ed-acde-289111e5001c",
     "description": "Detailed budget credits 2019"},
    {"name": "credito_2018", "id": "61f9adca-8c7a-4e4e-b8b7-8933f425a257",
     "description": "Detailed budget credits 2018"},
    {"name": "credito_2017", "id": "85fd7ca2-745d-439d-aa51-20ec24d11800",
     "description": "Detailed budget credits 2017"},
    {"name": "credito_2016", "id": "3b339839-bf94-4a91-94a6-39a2063692cd",
     "description": "Detailed budget credits 2016"},
    {"name": "credito_2015", "id": "90c441cc-c33d-4a92-a8b6-c8df62b59106",
     "description": "Detailed budget credits 2015"},
    {"name": "credito_2014", "id": "79678ede-70d0-43bf-9cc8-ecfdc3119e34",
     "description": "Detailed budget credits 2014"},
    {"name": "credito_2013", "id": "cd890cac-2070-4828-8fe2-52381b1559ab",
     "description": "Detailed budget credits 2013"},
    {"name": "credito_2012", "id": "a5180ee3-67f3-4427-9a35-1e62628ea606",
     "description": "Detailed budget credits 2012"},
    {"name": "credito_2011", "id": "8ca3260d-932e-43aa-a775-1c5abadf80eb",
     "description": "Detailed budget credits 2011"},
    {"name": "organismos", "id": "a8742e97-5da5-4c46-9393-b31640f61ee0",
     "description": "National budget organizations"},
    {"name": "unidades_ejecutoras", "id": "24beb1d3-b1d6-498f-b2ee-8c365a2c4157",
     "description": "Executing units"},
    {"name": "areas_programaticas", "id": "4c1bf56a-6e16-45f0-b670-505056ece39d",
     "description": "Programmatic areas"},
    {"name": "programas", "id": "25d142fe-e10b-43c2-aa7d-5761791f6c81",
     "description": "Programs"},
]

print(f"{len(BUDGET_RESOURCES)} budget resources to download via datastore dump")

In [ ]:
budget_dataframes = {}

with httpx.Client() as client:
    for res in BUDGET_RESOURCES:
        url = f"{DATASTORE_DUMP}/{res['id']}?bom=True&format=csv"
        blob_name = f"raw/transparency/{res['name']}.parquet"
        print(f"Downloading: {res['name']} — {res['description']}")

        try:
            r = client.get(url, follow_redirects=True, timeout=90.0)
            r.raise_for_status()
            df = read_csv_robust(r.content)
        except Exception as exc:
            print(f"  SKIP: {exc}")
            continue

        if df.is_empty():
            print(f"  SKIP: empty")
            continue

        upload_df_to_gcs(df, blob_name)
        budget_dataframes[res["name"]] = df
        print(f"  OK {df.shape[0]} rows, {df.shape[1]} cols")

print(f"\nUploaded {len(budget_dataframes)} budget datasets to GCS")

---
## Stream C — PDF Scraping

Scrape PDF links from the OPP budget page, download them, and upload raw PDFs to GCS.

In [ ]:
BASE_URL = "https://www.opp.gub.uy/es/presupuesto-nacional"
FALLBACK_URLS = [
    "https://www.opp.gub.uy/es/direccion-presupuestos",
    "https://transparenciapresupuestaria.opp.gub.uy/inicio/presupuesto-nacional",
]
DELAY_SECONDS = 2.5

pdf_links = []

with httpx.Client(headers={"User-Agent": "OPP-Budget-Research/1.0 (academic)"}, timeout=30.0) as client:
    # Try main URL + fallbacks
    for url in [BASE_URL] + FALLBACK_URLS:
        try:
            resp = client.get(url, follow_redirects=True, timeout=30.0)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, "lxml")
            for anchor in soup.find_all("a", href=True):
                href = anchor["href"]
                if href.endswith(".pdf") and "/sites/default/files/" in href:
                    full_url = urljoin(url, href)
                    if full_url not in pdf_links:
                        pdf_links.append(full_url)
            if pdf_links:
                print(f"Found {len(pdf_links)} PDF links from {url}")
                break
        except httpx.HTTPError as exc:
            print(f"  {url} — {exc}")
            continue

    if not pdf_links:
        # Fallback: list existing PDFs from GCS (already cached from previous runs)
        print("OPP site unavailable — using cached PDFs from GCS")
        cached_blobs = list(bucket.list_blobs(prefix="raw/pdfs/"))
        cached_pdfs = [b for b in cached_blobs if b.name.endswith(".pdf")]
        print(f"Found {len(cached_pdfs)} cached PDFs in GCS (skipping scrape)")
    else:
        for link in pdf_links:
            print(f"  {link.split('/')[-1]}")

In [ ]:
downloaded_pdfs = []

if pdf_links:
    with httpx.Client(headers={"User-Agent": "OPP-Budget-Research/1.0 (academic)"}) as client:
        for i, url in enumerate(pdf_links):
            filename = url.split("/")[-1]
            blob_name = f"raw/pdfs/{filename}"

            blob = bucket.blob(blob_name)
            if blob.exists():
                print(f"  [{i+1}/{len(pdf_links)}] CACHED {filename}")
                downloaded_pdfs.append(filename)
                continue

            try:
                r = client.get(url, follow_redirects=True, timeout=120.0)
                r.raise_for_status()
            except httpx.HTTPError as exc:
                print(f"  [{i+1}/{len(pdf_links)}] SKIP {filename}: {exc}")
                continue

            blob.upload_from_string(r.content, content_type="application/pdf")
            size_mb = len(r.content) / (1024 * 1024)
            downloaded_pdfs.append(filename)
            print(f"  [{i+1}/{len(pdf_links)}] OK {filename} ({size_mb:.1f} MB)")

            if i < len(pdf_links) - 1:
                time.sleep(DELAY_SECONDS)

    print(f"\nUploaded {len(downloaded_pdfs)} PDFs to GCS")
else:
    # OPP site down — use cached GCS PDFs
    cached_blobs = list(bucket.list_blobs(prefix="raw/pdfs/"))
    downloaded_pdfs = [b.name.split("/")[-1] for b in cached_blobs if b.name.endswith(".pdf")]
    print(f"Using {len(downloaded_pdfs)} cached PDFs from GCS (OPP site unavailable)")

---
## Ingestion Summary

In [ ]:
print("=" * 60)
print("GCS BUCKET CONTENTS")
print("=" * 60)
for prefix in ["raw/ckan/", "raw/transparency/", "raw/pdfs/", "raw/cgn/"]:
    blobs = list(bucket.list_blobs(prefix=prefix))
    files = [b for b in blobs if not b.name.endswith("/")]
    total_mb = sum(b.size for b in files) / (1024 * 1024)
    print(f"\n{prefix}")
    print(f"  Files: {len(files)}  |  Total: {total_mb:.1f} MB")
    for b in files[:5]:
        print(f"    {b.name.split('/')[-1]}  ({b.size / 1024:.0f} KB)")
    if len(files) > 5:
        print(f"    ... and {len(files) - 5} more")

---
## Stream D — CGN SIIF Budget Execution (Reference Data)

Official per-inciso budget execution from the Contaduría General de la Nación (SIIF system).
The CGN site (`cgn.gub.uy`) uses JSF/XHTML which is not API-friendly, so this data is embedded
as a validated reference table from official published figures.

**Source:** https://www.cgn.gub.uy/siifEjecucionPresupuestalPresentacion/
**Columns:** fiscal_year, inciso, denominacion_inciso, credito_vigente, obligado_ejecutado

In [ ]:
# ── CGN SIIF Budget Execution: validated reference data ──
# Source: https://www.cgn.gub.uy/siifEjecucionPresupuestalPresentacion/
# Note: Incisos 28 (BPS) and 30 (Deuda Publica) are not in CGN data.
#       Inciso 36 (Ministerio de Ambiente) only exists from 2023.

cgn_2020 = pl.DataFrame({
    "fiscal_year": [2020] * 31,
    "inciso": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 29, 31, 32, 33],
    "denominacion_inciso": [
        "Poder Legislativo",
        "Presidencia de la Republica",
        "Ministerio de Defensa Nacional",
        "Ministerio del Interior",
        "Ministerio de Economia y Finanzas",
        "Ministerio de Relaciones Exteriores",
        "Ministerio de Ganaderia, Agricultura y Pesca",
        "Ministerio de Industria, Energia y Mineria",
        "Ministerio de Turismo",
        "Ministerio de Transporte y Obras Publicas",
        "Ministerio de Educacion y Cultura",
        "Ministerio de Salud Publica",
        "Ministerio de Trabajo y Seguridad Social",
        "Min. de Vivienda, Ord. Terr. y Medio Ambiente",
        "Ministerio de Desarrollo Social",
        "Poder Judicial",
        "Tribunal de Cuentas",
        "Corte Electoral",
        "Tribunal de lo Contencioso Administrativo",
        "Intereses de la Deuda Publica",
        "Subsidios y Subvenciones",
        "Transferencias Financieras",
        "Partidas a Reaplicar",
        "Diversos Creditos",
        "Adm. Nacional de Educacion Publica",
        "Universidad de la Republica",
        "Inst. del Nino y Adolescente del Uruguay",
        "Adm. de los Servicios de Salud del Estado",
        "Univ. Tecnologica del Uruguay",
        "Inst. Uruguayo de Meteorologia",
        "Fiscalia General de la Nacion",
    ],
    "credito_vigente": [
        7240842799.0, 5073514474.0, 27469864232.0, 39416667934.0, 14319631405.0,
        4015471763.0, 4559424508.0, 1403777822.0, 715705822.0, 14288270722.0,
        4858075584.0, 4408674608.0, 1386484394.0, 11005024219.0, 11521524579.0,
        7564701597.0, 722642795.0, 1486406727.0, 208310525.0, 66078993218.0,
        14150014342.0, 152913918478.0, 13687953648.0, 155103874231.0, 77485191423.0,
        19212671501.0, 15508271719.0, 43464421665.0, 942304337.0, 325241340.0,
        1493899709.0,
    ],
    "obligado_ejecutado": [
        5279521382.0, 3814956786.0, 24149087988.0, 35457115879.0, 12051177509.0,
        3799739331.0, 3574903233.0, 1054211236.0, 533172362.0, 12124620553.0,
        3813211009.0, 4009917476.0, 1084108083.0, 10666596656.0, 10638777822.0,
        7261774301.0, 689889107.0, 1336342575.0, 191604392.0, 66078993218.0,
        12800086896.0, 152872544353.0, 0.0, 152975240866.0, 76004392925.0,
        18923514108.0, 14737503484.0, 43262431687.0, 911457066.0, 293442075.0,
        1476885222.0,
    ],
}).with_columns([
    pl.col("fiscal_year").cast(pl.Int64),
    pl.col("inciso").cast(pl.Int64),
    pl.col("credito_vigente").cast(pl.Float64),
    pl.col("obligado_ejecutado").cast(pl.Float64),
])

# 2020 has 31 incisos; add remaining 3 (34, 35 not included above)
cgn_2020_extra = pl.DataFrame({
    "fiscal_year": [2020, 2020],
    "inciso": [34, 35],
    "denominacion_inciso": [
        "Junta de Transparencia y Etica Publica",
        "Inst. Nacional de Inclusion Social Adolescente",
    ],
    "credito_vigente": [38894539.0, 2454224667.0],
    "obligado_ejecutado": [28385170.0, 2227649909.0],
}).with_columns([
    pl.col("fiscal_year").cast(pl.Int64),
    pl.col("inciso").cast(pl.Int64),
    pl.col("credito_vigente").cast(pl.Float64),
    pl.col("obligado_ejecutado").cast(pl.Float64),
])
cgn_2020 = pl.concat([cgn_2020, cgn_2020_extra]).sort("inciso")

cgn_2023 = pl.DataFrame({
    "fiscal_year": [2023] * 33,
    "inciso": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 29, 31, 32, 33, 34, 35],
    "denominacion_inciso": [
        "Poder Legislativo",
        "Presidencia de la Republica",
        "Ministerio de Defensa Nacional",
        "Ministerio del Interior",
        "Ministerio de Economia y Finanzas",
        "Ministerio de Relaciones Exteriores",
        "Ministerio de Ganaderia, Agricultura y Pesca",
        "Ministerio de Industria, Energia y Mineria",
        "Ministerio de Turismo",
        "Ministerio de Transporte y Obras Publicas",
        "Ministerio de Educacion y Cultura",
        "Ministerio de Salud Publica",
        "Ministerio de Trabajo y Seguridad Social",
        "Min. de Vivienda y Ordenamiento Territorial",
        "Ministerio de Desarrollo Social",
        "Poder Judicial",
        "Tribunal de Cuentas",
        "Corte Electoral",
        "Tribunal de lo Contencioso Administrativo",
        "Intereses de la Deuda Publica",
        "Subsidios y Subvenciones",
        "Transferencias Financieras",
        "Partidas a Reaplicar",
        "Diversos Creditos",
        "Adm. Nacional de Educacion Publica",
        "Universidad de la Republica",
        "Inst. del Nino y Adolescente del Uruguay",
        "Adm. de los Servicios de Salud del Estado",
        "Univ. Tecnologica del Uruguay",
        "Inst. Uruguayo de Meteorologia",
        "Fiscalia General de la Nacion",
        "Junta de Transparencia y Etica Publica",
        "Inst. Nacional de Inclusion Social Adolescente",
    ],
    "credito_vigente": [
        10098862118.0, 6051796944.0, 36310669177.0, 52858390955.0, 16775601521.0,
        3995712251.0, 5832204671.0, 1668911791.0, 792974686.0, 20221574421.0,
        6285713176.0, 6732498618.0, 1685648261.0, 13245005667.0, 27265414248.0,
        9548923007.0, 951829660.0, 1776141727.0, 281484023.0, 73907192208.0,
        14973310243.0, 175447763317.0, 11022618871.0, 189107355062.0, 100670163446.0,
        24615768478.0, 21054448441.0, 58518751219.0, 1507216230.0, 380560082.0,
        1918257242.0, 57424491.0, 2792529898.0,
    ],
    "obligado_ejecutado": [
        6311365940.0, 4612871788.0, 31852600224.0, 48103531007.0, 13626669312.0,
        3697355212.0, 4439736593.0, 1245998962.0, 698595270.0, 17677026456.0,
        4929069351.0, 6195574422.0, 1236410476.0, 12750650238.0, 26601548445.0,
        8965410887.0, 891692280.0, 1662334714.0, 240625592.0, 73907192208.0,
        14955561545.0, 175447756096.0, 0.0, 185285900902.0, 97197734249.0,
        24116525071.0, 20657440924.0, 57580042042.0, 1501982756.0, 347225948.0,
        1828516588.0, 36799780.0, 2784766931.0,
    ],
}).with_columns([
    pl.col("fiscal_year").cast(pl.Int64),
    pl.col("inciso").cast(pl.Int64),
    pl.col("credito_vigente").cast(pl.Float64),
    pl.col("obligado_ejecutado").cast(pl.Float64),
])

# Add inciso 36 (Ministerio de Ambiente) — only exists in 2023
cgn_2023_extra = pl.DataFrame({
    "fiscal_year": [2023],
    "inciso": [36],
    "denominacion_inciso": ["Ministerio de Ambiente"],
    "credito_vigente": [1047285810.0],
    "obligado_ejecutado": [883541038.0],
}).with_columns([
    pl.col("fiscal_year").cast(pl.Int64),
    pl.col("inciso").cast(pl.Int64),
    pl.col("credito_vigente").cast(pl.Float64),
    pl.col("obligado_ejecutado").cast(pl.Float64),
])
cgn_2023 = pl.concat([cgn_2023, cgn_2023_extra]).sort("inciso")

# Combine and upload
cgn_all = pl.concat([cgn_2020, cgn_2023]).sort(["fiscal_year", "inciso"])

upload_df_to_gcs(cgn_2020, "raw/cgn/siif_ejecucion_2020.parquet")
upload_df_to_gcs(cgn_2023, "raw/cgn/siif_ejecucion_2023.parquet")
upload_df_to_gcs(cgn_all, "raw/cgn/siif_ejecucion_all.parquet")

print("CGN SIIF Budget Execution — uploaded to GCS")
print(f"  2020: {cgn_2020.shape[0]} incisos")
print(f"  2023: {cgn_2023.shape[0]} incisos")
print(f"  Combined: {cgn_all.shape[0]} rows")
print(f"\nSchema: {cgn_all.schema}")
print(f"\n2020 total credito_vigente: {cgn_2020['credito_vigente'].sum():,.0f}")
print(f"2020 total obligado_ejecutado: {cgn_2020['obligado_ejecutado'].sum():,.0f}")
print(f"2023 total credito_vigente: {cgn_2023['credito_vigente'].sum():,.0f}")
print(f"2023 total obligado_ejecutado: {cgn_2023['obligado_ejecutado'].sum():,.0f}")

---
## Exploratory Data Analysis

In [ ]:
all_dfs = {**ckan_dataframes, **budget_dataframes}

print(f"Datasets available for EDA: {len(all_dfs)}")
print()
for name, df in all_dfs.items():
    print(f"── {name} ──")
    print(f"   Shape: {df.shape}")
    print(f"   Columns: {df.columns}")
    print(f"   Nulls: {df.null_count().to_dict()}")
    print()

In [ ]:
# Preview each dataset
for name, df in all_dfs.items():
    print(f"\n{'=' * 60}")
    print(f"{name}")
    print(f"{'=' * 60}")
    print(df.head(5))
    print()
    print("Schema:")
    for col, dtype in zip(df.columns, df.dtypes):
        print(f"  {col}: {dtype}")

In [ ]:
# Statistical summary of numeric columns
for name, df in all_dfs.items():
    numeric_cols = [c for c, t in zip(df.columns, df.dtypes) if t.is_numeric()]
    if numeric_cols:
        print(f"\n── {name}: numeric summary ──")
        print(df.select(numeric_cols).describe())

In [ ]:
# Check for budget-related columns across all datasets
budget_keywords = ["inciso", "presupuest", "credito", "ejecucion", "gasto",
                   "monto", "anio", "año", "fiscal", "categoria", "programa"]

print("Budget-related columns found:")
for name, df in all_dfs.items():
    matches = [c for c in df.columns
               if any(kw in c.lower() for kw in budget_keywords)]
    if matches:
        print(f"  {name}: {matches}")

In [ ]:
print("\n Ingestion & EDA complete.")
print(f"   CKAN package datasets:   {len(ckan_dataframes)}")
print(f"   Budget datastore dumps:  {len(budget_dataframes)}")
print(f"   PDFs uploaded:           {len(downloaded_pdfs)}")